In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
from collections import defaultdict

: 

In [ ]:

root = Path("/home/linux/Desktop/Dataset/")  # replace with your top-level directory
files = list(root.rglob("*.jsonl"))
total_files = len(files) # Count the number of files
records = []
file_count = 0
line_count = 0
bad_lines = 0
total_lines_to_process = 0

# Iterate through each file in the list 'files'
for file in files:
    # Open each file in read mode with UTF-8 encoding
    with file.open("r", encoding="utf-8") as f:
        # Count each line in the current file
        for line in f:
            total_lines_to_process += 1  # Increment the line counter

# Print out the total number of files and the total lines to process
print(f"Total files: {total_files}, Total lines to process: {total_lines_to_process}")

In [ ]:
for p in files:
    file_count += 1
    print(f"Processing file {file_count}/{total_files}: {p}")
    with p.open("r", encoding="utf-8") as f:
        for i, raw in enumerate(f, start=1):
            raw = raw.strip()
            if not raw:
                continue
            try:
                records.append(json.loads(raw))
                line_count += 1
            except json.JSONDecodeError:
                bad_lines += 1
    print(f"  -> lines read: {i}, valid records so far: {line_count}, bad lines so far: {bad_lines}")

df = pd.DataFrame.from_records(records)
print(f"Finished. Files: {total_files}, Records: {len(df)}, Bad lines: {bad_lines}")

In [ ]:
df.info()
df.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Label distribution
label_counts = df['label'].value_counts()
axes[0,0].pie(label_counts.values, labels=['Benign' if x==0 else 'Malware' for x in label_counts.index], 
              autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'])
axes[0,0].set_title('Malware vs Benign Distribution')

# File type distribution (top 10)
df['file_type'].value_counts().head(10).plot(kind='barh', ax=axes[0,1], color='#3498db')
axes[0,1].set_title('Top 10 File Types')

# Family distribution (top 10, excluding NaN)
df['family'].value_counts().head(10).plot(kind='barh', ax=axes[1,0], color='#9b59b6')
axes[1,0].set_title('Top 10 Malware Families')

# Detection ratio distribution
df['detection_ratio'].value_counts().head(10).plot(kind='bar', ax=axes[1,1], color='#f39c12')
axes[1,1].set_title('Detection Ratio Distribution')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('malware_overview.png', dpi=150, bbox_inches='tight')
plt.show()